In [1]:
import torch
from torch import nn
from torch.nn import functional as F
from d2l import torch as d2l


#               ┌──────── Shortcut ────────┐
#               │                          ▼
# x ─→ Weight Layers ─→ g(x) ───────────→ (+) ─→ Activation

In [2]:
# Residual Block

class Residual(nn.Module):
    def __init__(
        self,
        num_channels: int,
        use_1x1conv: bool = False,
        strides: int = 1,
    ) -> None:
        super().__init__()
        
        # Main Branch:
        # - kernel: 3x3 Convolution
        # - output shape 유지
        self.conv1 = nn.LazyConv2d(
            out_channels=num_channels,
            kernel_size=3,
            padding=1,
            stride=strides,
        )
        self.conv2 = nn.LazyConv2d(
            out_channels=num_channels,
            kernel_size=3,
            padding=1,
        )
        
        # Batch Normalization:
        self.bn1 = nn.LazyBatchNorm2d()
        self.bn2 = nn.LazyBatchNorm2d()
        
        # Shortcut Branch: 
        # - kernel: 1x1 Convolution (Projection)
        # - output channel를 Y와 맞추기
        self.conv3: nn.Module | None = (
            nn.LazyConv2d(
                out_channels=num_channels,
                kernel_size=1,
                stride=strides,
            )
            if use_1x1conv
            else None
        )
        
        
    def forward(
        self,
        X: torch.Tensor,
    ) -> torch.Tensor:
        
        # Main Branch
        Y = F.relu(
            self.bn1(
                self.conv1(X)
            )
        )
        Y = self.bn2(
            self.conv2(Y)
        )
        
        # Shortcut Branch
        shortcut = X

        if self.conv3 is not None:
            shortcut = self.conv3(
                shortcut,
            )
        
        # Residual Addition 후 Activation
        return F.relu(
            Y + shortcut
        )

In [5]:
# Identity Shortcut: g(x)와 동일한 f(x)

# [B, C, H, W]
X = torch.randn(
    4,
    3,
    6,
    6,
)

identity_block = Residual(
    num_channels=3,
    use_1x1conv=False,   
)

with torch.no_grad():
    Y_identity = identity_block(
        X
    )
    
print(
    "Input shape:",
    tuple(X.shape),
)

# num_channel=3, strides=1이므로
print(
    "Output shape:",
    tuple(Y_identity.shape),
)
    

Input shape: (4, 3, 6, 6)
Output shape: (4, 3, 6, 6)


In [4]:
# Projection Shortcut

projection_block = Residual(
    num_channels=6,
    use_1x1conv=True,
    strides=2,
)

with torch.no_grad():
    Y_projection = projection_block(
        X
    )
    
print(
    "Input shape:",
    tuple(X.shape),
)

print(
    "Output shape:",
    tuple(Y_projection.shape),
)

Input shape: (4, 3, 6, 6)
Output shape: (4, 6, 3, 3)
